# FDGAN: train on your data and sample

This notebook runs the FDGAN code of the paper end to end on Colab:

1. **Install** the repository.
2. **Data**: either the Traffic Signs crops of our dataset release, or your own images with YOLO labels (any size;
   they are turned into 256 px object-centric crops with one box each).
3. **Train** FDGAN on those crops, from scratch or starting from a checkpoint, with the paper's flags. Checkpoints go
   to your Google Drive, so a run survives a Colab disconnect and continues where it stopped.
4. **Sample** from the model you trained, or from one of our checkpoints, and look at the images with their boxes.

Use a GPU runtime (Runtime > Change runtime type). The paper's runs are 200,000 iterations at batch 16 on an A100,
about a day each; on a Colab T4 the same recipe runs at a smaller batch and takes correspondingly longer, in several
sessions. Every flag below is the one from `jobs/` and `tools/make_sbatch_files.py`; only the batch size and the
horizon are yours to set.

In [ ]:
# 1. install (about two minutes: mpi4py builds from source)
!apt-get -qq install -y libopenmpi-dev openmpi-bin > /dev/null
!pip -q install blobfile mpi4py
!git clone -q https://github.com/Latzi/focused-diffusion-gan.git /content/focused-diffusion-gan
%cd /content/focused-diffusion-gan
!pip -q install -e . > /dev/null
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Data

FDGAN trains on square crops with one bounding box each. Run **2a** for the Traffic Signs crops of our release, or
**2b** for your own data. Both end with two folders, `DATA_DIR` (the 256 px crops) and `BB_DIR` (one JSON box per crop
in x-y-width-height pixels), which the training and sampling cells use.

In [ ]:
# 2a. the Traffic Signs crops of our dataset release (3,009 crops, about 40 MB)
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/Latzi/object-centric-low-data-datasets.git /content/ocld
!cd /content/ocld && git sparse-checkout set traffic_signs/Train_Data > /dev/null
!python tools/make_box_json_from_release.py --traffic_signs /content/ocld/traffic_signs --out_boxes /content/data/traffic_bb --out_images /content/data/traffic_img
DATA_DIR, BB_DIR = "/content/data/traffic_img", "/content/data/traffic_bb"

In [ ]:
# 2b. your own images with YOLO labels (skip if you ran 2a)
# IMAGES: a folder of images (any size, searched recursively). LABELS: YOLO text files of the same names,
# one line per box "class x_center y_center width height" normalised to the image. Both can be on your Drive.
IMAGES = "/content/drive/MyDrive/my_dataset/images"
LABELS = "/content/drive/MyDrive/my_dataset/labels"

import os
if (IMAGES.startswith("/content/drive") or LABELS.startswith("/content/drive")) and not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
!python tools/prepare_dataset.py --images "$IMAGES" --labels "$LABELS" --out_images /content/data/my_img --out_boxes /content/data/my_bb --size 256 --rule largest --window object
DATA_DIR, BB_DIR = "/content/data/my_img", "/content/data/my_bb"

## 3. Train

Set the four values in the next cell and run the cell after it.

- `OUT_DIR`: a folder on your Google Drive. Model, EMA, optimiser and discriminator checkpoints are written there
  every `SAVE_EVERY` iterations. If Colab disconnects, run the cell again: it resumes from the newest checkpoint in
  that folder, discriminator included.
- `ITERATIONS`: the training horizon. The learning rate decays linearly to zero at this iteration and the run stops.
  The paper uses 200,000; the adversarial stage starts at iteration 40,000 (`GAN_WARMUP`), so a run shorter than that
  is plain ADM with the mask channel (FDGAN\* in the paper).
- `BATCH` and `MICRO`: the paper uses 16 and 8 (39 GB on an A100). On a 16 GB T4 use 4 and 2, or 8 and 2 on a 40 GB
  A100 runtime. Smaller batches change the recipe; say so if you report the result.
- `INIT_CHECKPOINT`: empty for training from scratch (Cityscapes and Traffic Signs in the paper), or a `.pt` to start
  from (the COCO rows start from an LSUN-trained ADM checkpoint). A 3-channel ADM checkpoint is widened to the four
  channels on load; one of our 4-channel checkpoints loads as is. Starting from a checkpoint restarts the iteration
  count at zero.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/FDGAN/run_traffic"   # checkpoints and logs, on your Drive
ITERATIONS = 200000      # the paper's horizon; the run stops here
BATCH, MICRO = 4, 2      # 16, 8 in the paper (A100); 4, 2 fits a T4
INIT_CHECKPOINT = ""     # "" = from scratch; or a .pt file to start from
GAN_WARMUP = 40000       # adversarial stage starts here (the paper's value)
SAVE_EVERY = 2000        # checkpoint interval (10,000 in the paper); smaller is safer on Colab

In [ ]:
# 3. train (re-run this cell to continue after a disconnect)
import os, glob, re, shutil, subprocess
if OUT_DIR.startswith("/content/drive") and not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)
ckpts = sorted(glob.glob(os.path.join(OUT_DIR, "model*.pt")), key=lambda p: int(re.search(r"model(\d+)\.pt$", p).group(1)))
if ckpts:
    resume, script = ckpts[-1], "scripts/image_train_Run2_3_10_FDGAN_COCO_LateBand.py"
    print("resuming from", resume)
elif INIT_CHECKPOINT:
    os.makedirs("/content/init", exist_ok=True)
    resume = "/content/init/init.pt"          # a name without a step number, so the iteration count starts at 0
    if not os.path.exists(resume): shutil.copy(INIT_CHECKPOINT, resume)
    script = "scripts/image_train_Run2_3_10_FDGAN_COCO_LateBand_lsun.py"   # the copy of the script used for runs that start from a checkpoint
    print("starting from", INIT_CHECKPOINT)
else:
    resume, script = "", "scripts/image_train_Run2_3_10_FDGAN_COCO_LateBand.py"
    print("starting from scratch")
cmd = (f"python {script} --data_dir {DATA_DIR} --bb_dir {BB_DIR} --bb_is_xywh True "
       f"--log_dir {OUT_DIR}/logs --checkpoint_dir {OUT_DIR} --image_save_dir {OUT_DIR}/debug_images "
       f"--lr 1e-4 --batch_size {BATCH} --microbatch {MICRO} --lr_anneal_steps {ITERATIONS} --save_interval {SAVE_EVERY} "
       f"--gan_warmup_steps {GAN_WARMUP} --image_size 256 --num_channels 128 --num_res_blocks 2 --num_heads 1 --attention_resolutions 16 --use_scale_shift_norm False --resblock_updown False --learn_sigma True --class_cond False --dropout 0.0 --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --use_fp16 True --fp16_scale_growth 1e-3 --ema_rate 0.9999 --log_interval 10 --bb_mask_key bb_mask --mask_gain 50.0 --mask_init_mode copy_mean --mask_init_scale 0.10 --tmin_start 820 --tmin_end 700 --band_width 120 --tmax_cap 950 --band_ramp_steps 50000 --diffusion_in_band_weight 0.7 --gan_initial_scale 1e-4 --gan_final_scale 1e-2 --l1_initial_scale 5e-2 --l1_final_scale 5e-2 --lambda_enhanced 5e-2 --color_stats_scale 0.05 --sat_penalty_scale 0.01 --sat_threshold 0.95 --noise_option extra_gan_only --k_focus 1.75 --max_gan_per_microbatch 4 --use_mask_adapter True --ndf 128 --n_layers_D 3 --use_sigmoid False --lr_D 1e-5 --beta1_D 0.5 --beta2_D 0.999 --save_discriminators True --resume_discriminators True --g_adv_warmup_steps_if_no_d_ckpt 2000 --bb_xyxy_end_inclusive False --image_save_every 1000 --use_bb_mask_cond True "
       + (f"--resume_checkpoint {resume}" if resume else ""))
print(cmd); print()
!{cmd}

## 4. Sample

`CHECKPOINT` defaults to the newest EMA checkpoint in `OUT_DIR`. To sample from one of our trained models instead,
put its `ema_0.9999_200000.pt` on your Drive and set the path, or give a direct download link. The flags are the
paper's: mask channel on, the perturbed-input prediction inside the band, 500 ancestral steps, mask guidance
`CFG_SCALE` (the paper reports 1.5 and 2.0). Boxes come from `BB_DIR` in sorted order, so sample i always gets box i.

In [ ]:
CHECKPOINT = ""      # "" = newest EMA checkpoint in OUT_DIR; or a path / download link to an ema_*.pt
NUM, CFG_SCALE = 16, 2.0

import os, glob, re, urllib.request
if not CHECKPOINT:
    emas = sorted(glob.glob(os.path.join(OUT_DIR, "ema_*.pt")), key=lambda p: int(re.search(r"_(\d+)\.pt$", p).group(1)))
    assert emas, f"no EMA checkpoint in {OUT_DIR} yet"
    CKPT = emas[-1]
elif CHECKPOINT.startswith("http"):
    os.makedirs("/content/ckpt", exist_ok=True)
    CKPT = "/content/ckpt/" + os.path.basename(CHECKPOINT.split("?")[0])
    if not os.path.exists(CKPT): urllib.request.urlretrieve(CHECKPOINT, CKPT)
else:
    CKPT = CHECKPOINT
    if not os.path.exists(CKPT):
        from google.colab import drive; drive.mount("/content/drive")
assert os.path.exists(CKPT), f"not found: {CKPT}"
print("sampling from", CKPT, f"({os.path.getsize(CKPT)/1e6:.0f} MB)")
OUT_SAMPLES = "/content/samples"
cmd = (f"python scripts/image_sample_fdgan_4ch.py --model_path {CKPT} --bb_dir {BB_DIR} --out_dir {OUT_SAMPLES} "
       f"--num_samples {NUM} --batch_size 8 --seed 17 --timestep_respacing 500 --mask_cfg_scale {CFG_SCALE} --bb_is_xywh True --image_size 256 --num_channels 128 --num_res_blocks 2 --num_heads 1 --attention_resolutions 16 --use_scale_shift_norm False --resblock_updown False --learn_sigma True --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --use_bb_mask_cond True --mask_gain 50.0 --mask_mode always --roi_bump_mode predict --k_focus 1.75 --bump_tmin_rev 700 --bump_tmax_rev 820 --bb_xyxy_end_inclusive False")
!{cmd}

In [ ]:
# 5. look at the samples
import json, math, matplotlib.pyplot as plt
from PIL import Image, ImageDraw
meta = json.load(open(f"{OUT_SAMPLES}/meta.json"))
n = len(meta["samples"]); cols = 8; rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 2.7 * rows))
for ax, rec in zip(axes.flat, meta["samples"]):
    im = Image.open(f"{OUT_SAMPLES}/{rec['file']}").convert("RGB")
    x0, y0, x1, y1 = rec["bb_xyxy"]
    ImageDraw.Draw(im).rectangle([x0, y0, x1 - 1, y1 - 1], outline=(255, 40, 40), width=2)
    ax.imshow(im); ax.set_title(rec["file"][-10:-4], fontsize=8); ax.axis("off")
for ax in list(axes.flat)[n:]: ax.axis("off")
plt.tight_layout(); plt.show()
print("cost:", meta["cost"])

## Notes

- The training logs are in `OUT_DIR/logs/progress.csv`: `diffusion_loss` for every step, and once the band is open
  `gan_active_frac`, `d_loss`, `g_adv`, `g_l1`, `g_roi` and the others listed in `docs/CODE_GUIDE.md`.
  `OUT_DIR/debug_images` gets a grid of in-band samples (perturbed input, clean-image prediction, box) every 1,000
  iterations.
- To train the matched control FDGAN\* (no adversarial stage), set `GAN_WARMUP = 999999999`. For plain ADM also add
  `--use_bb_mask_cond False` to the command. The ablation rows of the paper are the other flag changes listed in
  `tools/make_sbatch_files.py`.
- Boxes stored as corners (x1, y1, x2, y2) instead of x-y-width-height: drop `--bb_is_xywh True` from both commands.
  Both converters in `tools/` write x-y-width-height.